# CodeAlpha Project: Car Price Prediction
**Beginner-friendly, start-to-finish notebook**

## What you will build
A machine-learning model that estimates a used car's selling price from details such as year, showroom price, kilometers driven, fuel type, seller type, transmission, and previous owners.

**Important:** This is a learning project. The model's predictions are estimates, not guaranteed market prices. Results depend on the dataset and its quality.

## Project roadmap
1. Import libraries.
2. Load the dataset.
3. Understand and clean the data.
4. Explore patterns with charts.
5. Prepare features and target.
6. Train and evaluate a model.
7. Test a sample prediction.
8. Save the trained model.
9. Review professional improvements and interview questions.

### How to run this notebook
- Easiest option: open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and upload this `.ipynb` file.
- Run each cell from top to bottom using the play button.
- Internet is needed for the default dataset download. If it fails, upload a compatible CSV when prompted in the data-loading cell.

## Step 1 — Install/import the tools
We use pandas for tables, matplotlib/seaborn for charts, and scikit-learn to train and evaluate a model. Most Colab environments already have these packages.

In [ ]:
# If needed, uncomment and run this line:
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
print("Libraries imported successfully!")

## Step 2 — Load the dataset
The default URL points to a commonly used teaching dataset. If the download fails, the notebook asks you to upload `car data.csv` (or another compatible CSV).

Expected columns include `Year`, `Selling_Price`, `Present_Price`, `Kms_Driven`, `Fuel_Type`, `Seller_Type`, `Transmission`, and `Owner`. Column names are standardized below.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/krishnaik06/Car-Price-Prediction/master/car%20data.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Dataset downloaded from the public URL.")
except Exception as e:
    print("Could not download the default dataset:", e)
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise ValueError("No CSV file was uploaded.")
        df = pd.read_csv(csv_files[0])
    except ImportError:
        local_csv = Path("car data.csv")
        if not local_csv.exists():
            raise FileNotFoundError(
                "Place a compatible CSV named 'car data.csv' beside this notebook, "
                "or run it in Google Colab and upload a CSV when prompted."
            )
        df = pd.read_csv(local_csv)

print("Rows and columns:", df.shape)
display(df.head())

## Step 3 — Inspect and clean the data
Always inspect column names, data types, missing values, and duplicates before modeling. We normalize common column names and check that the target column `Selling_Price` exists.

**Note:** This example dataset records prices in lakhs (Indian rupees, in the dataset's original convention). Keep the same units when interpreting predictions.

In [ ]:
# Standardize column names: remove surrounding spaces and use consistent spelling
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]

# Some copies use slightly different target/feature names.
rename_map = {
    "Selling_Price": "Selling_Price",
    "Present_Price": "Present_Price",
    "Kms_Driven": "Kms_Driven",
    "Fuel_Type": "Fuel_Type",
    "Seller_Type": "Seller_Type",
    "Transmission": "Transmission",
    "Owner": "Owner",
}
df = df.rename(columns=rename_map)

print("Columns:", df.columns.tolist())
display(df.info())
display(df.isna().sum().sort_values(ascending=False))
print("Duplicate rows:", df.duplicated().sum())

if "Selling_Price" not in df.columns:
    raise ValueError(
        "This CSV needs a 'Selling_Price' target column. "
        "Please use the suggested teaching dataset or adapt the target name in this cell."
    )

# Remove exact duplicate records
df = df.drop_duplicates().copy()

# Convert target and common numeric features to numeric where possible
for col in ["Selling_Price", "Present_Price", "Kms_Driven", "Year", "Owner"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Rows without a target cannot be used for supervised training
df = df.dropna(subset=["Selling_Price"])
df = df[df["Selling_Price"] >= 0].copy()

print("Shape after basic cleaning:", df.shape)
display(df.head())

## Step 4 — Explore the data with charts
Charts help you understand patterns before training. These plots are descriptive; they do not prove that one feature causes a price change.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Selling_Price"], kde=True)
plt.title("Distribution of Selling Prices")
plt.xlabel("Selling price (dataset units)")
plt.tight_layout()
plt.show()

if "Year" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="Year", y="Selling_Price")
    plt.title("Selling Price vs. Year")
    plt.tight_layout()
    plt.show()

if "Kms_Driven" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="Kms_Driven", y="Selling_Price")
    plt.title("Selling Price vs. Kilometers Driven")
    plt.tight_layout()
    plt.show()

if "Fuel_Type" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="Fuel_Type", y="Selling_Price")
    plt.title("Selling Price by Fuel Type")
    plt.tight_layout()
    plt.show()

## Step 5 — Select features and prepare the model
We predict `Selling_Price`. We remove `Car_Name` because it can have many unique values in this small teaching dataset; you can investigate it later as an improvement. Missing numeric values are filled with the median; missing text values are filled with the most common category. Text features are converted to numeric indicators using one-hot encoding.

We split data into training and test sets. The test set is held back to evaluate predictions on rows the model did not train on.

In [ ]:
target = "Selling_Price"
drop_columns = [target]
if "Car_Name" in df.columns:
    drop_columns.append("Car_Name")

X = df.drop(columns=drop_columns)
y = df[target]

# Keep only rows with at least one feature and remove columns that are completely empty.
X = X.dropna(axis=1, how="all")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    ))
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Step 6 — Train the model
A Random Forest combines many decision trees and averages their predictions. It is a good baseline for tabular data and can model non-linear patterns.

In [ ]:
model.fit(X_train, y_train)
print("Model training complete!")

## Step 7 — Evaluate the model
We use three common regression metrics:

- **MAE**: average absolute prediction error; lower is better.
- **RMSE**: penalizes large errors more strongly; lower is better.
- **R²**: compares the model with predicting the test-set mean; closer to 1 is generally better. It can be negative if the model performs poorly.

Interpret all errors in the target's units. These metrics do not guarantee performance on cars from a different place, time, or market.

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE : {mae:.3f} target units")
print(f"RMSE: {rmse:.3f} target units")
print(f"R²  : {r2:.3f}")

results = pd.DataFrame({
    "Actual price": y_test.to_numpy(),
    "Predicted price": predictions,
    "Absolute error": np.abs(y_test.to_numpy() - predictions)
})
display(results.head(10))

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=predictions)
low = min(float(y_test.min()), float(predictions.min()))
high = max(float(y_test.max()), float(predictions.max()))
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual selling price")
plt.ylabel("Predicted selling price")
plt.title("Actual vs. Predicted Prices")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(y_test.to_numpy() - predictions, kde=True)
plt.axvline(0, linestyle="--")
plt.xlabel("Actual − predicted")
plt.title("Prediction Error Distribution")
plt.tight_layout()
plt.show()

## Step 8 — Predict one car
This cell uses one row from the test set, so the input has exactly the same feature columns the model expects. Later, you can replace it with details for a car you want to estimate.

In [ ]:
sample_car = X_test.iloc[[0]]
estimated_price = model.predict(sample_car)[0]

print("Car features:")
display(sample_car)
print(f"Estimated selling price: {estimated_price:.2f} target units")
print("Reminder: this is an estimate, not a guaranteed market price.")

## Step 9 — Save the model
The saved file includes preprocessing and the model, so future predictions use the same transformations. Only load joblib/pickle files that you trust.

In [ ]:
model_file = "car_price_model.joblib"
joblib.dump(model, model_file)
print(f"Saved model to: {model_file}")

# Test that the saved model can be loaded and used
loaded_model = joblib.load(model_file)
print("Loaded model prediction:", loaded_model.predict(sample_car)[0])

## Step 10 — Professional improvements
Try these after the basic notebook works:

1. Compare Random Forest with Linear Regression, Decision Tree, and Gradient Boosting.
2. Use cross-validation and hyperparameter tuning instead of relying on one split.
3. Add a feature such as `Car_Age = current year - Year`, while documenting which year is used.
4. Investigate outliers and unrealistic values with domain knowledge rather than deleting rows automatically.
5. Evaluate performance separately by fuel type, age group, or price range.
6. If using a newer dataset, verify currency, location, and time period; used-car prices change over time.
7. Create a small Streamlit app with a form for user input.
8. Add a README, requirements file, data source citation, screenshots, and limitations to GitHub.
9. Do not claim the model is production-ready without broader testing and monitoring.

## Step 11 — GitHub submission checklist
1. Create a GitHub repository named **`CodeAlpha_CarPricePrediction`** (or use the exact naming format your internship instructions require).
2. Upload this notebook and a `README.md`.
3. Include `requirements.txt` with the packages used.
4. Do not upload private data or secrets. Check the dataset's license/usage terms before redistributing it.
5. Run the notebook from a fresh session and confirm all cells work.
6. In your LinkedIn video, explain the problem, dataset, cleaning, charts, model, metrics, one prediction, and limitations.
7. Share the GitHub repository link in the internship submission form as instructed by your program.

Suggested `requirements.txt`:
```text
pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
jupyter
```

Suggested repository layout:
```text
CodeAlpha_CarPricePrediction/
├── CodeAlpha_Car_Price_Prediction.ipynb
├── README.md
├── requirements.txt
└── car_price_model.joblib   # optional; do not commit if large or restricted
```

## Step 12 — Interview questions and simple answers

**1. What is the goal of this project?**  
To estimate a used car's selling price from its available features.

**2. Is this classification or regression?**  
Regression, because the target is a continuous number.

**3. What is the target variable?**  
`Selling_Price`.

**4. Why split into train and test sets?**  
To estimate how well the model performs on unseen rows.

**5. What is data leakage?**  
When information unavailable at prediction time accidentally helps training or evaluation, making results look better than they should.

**6. Why use one-hot encoding?**  
Many machine-learning models need numbers, so categories are represented as indicator columns.

**7. What does MAE mean?**  
The average absolute difference between actual and predicted prices.

**8. What does R² mean?**  
It measures performance relative to a simple baseline that predicts the test-set average; higher is generally better.

**9. Why use a Random Forest?**  
It combines multiple decision trees and can learn non-linear relationships.

**10. What are the limitations?**  
The dataset may be small or old, and prices vary by location, condition, service history, and market timing.

**11. How would you improve the project?**  
Use more representative data, cross-validation, hyperparameter tuning, better features, and subgroup error analysis.

**12. Can the model guarantee the true price?**  
No. It provides an estimate based on patterns in the training data.

## Final checklist
- [ ] Notebook runs from the first cell to the last.
- [ ] Dataset source and target units are understood.
- [ ] Charts are labeled and explained.
- [ ] MAE, RMSE, and R² are reported honestly.
- [ ] One example prediction is shown.
- [ ] Model file saves and reloads.
- [ ] GitHub README explains the project and its limitations.
- [ ] LinkedIn video and internship submission are completed.

**Congratulations!** You now have a complete beginner-level machine-learning project that you can run, explain, and improve.